<a href="https://colab.research.google.com/github/nattynatasha-gif/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Setup and Data

In [1]:
import numpy as np
import csv


counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
 [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104], # Nairobi
 [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38], # Nakuru
 [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95], # Kisumu
 [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70], # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
 "9", "12", "", "45", "182", "96"]

months = ["January", "February", "March", "April", "May", "June",
 "July", "August", "September", "October", "November", "December"]





2. Clean Machakos Data

In [2]:

def clean_machakos(raw_list):
    """Convert valid rainfall strings to int; impute invalid with rounded mean of valid values."""
    valid = []
    for val in raw_list:
        try:
            valid.append(int(val.strip()))
        except ValueError:
            pass # skip invalid for the mean calculation

    if not valid:
        raise ValueError("No valid rainfall values found")

    mean_val = round(sum(valid) / len(valid))
    cleaned = []
    imputed_months = []

    for i, val in enumerate(raw_list):
        try:
            cleaned.append(int(val.strip()))
        except ValueError:
            cleaned.append(mean_val)
            imputed_months.append(months[i])

    print(f"Imputed months: {', '.join(imputed_months)}")
    return cleaned

machakos_clean = clean_machakos(machakos_raw)
print(f"Cleaned Machakos rainfall (mm): {machakos_clean}")

Imputed months: April, September
Cleaned Machakos rainfall (mm): [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


3. Create NumPy array

In [3]:

R = np.array(clean_rows + [machakos_clean])
print(f"Shape: {R.shape}")
print(f"Dtype: {R.dtype}")
print("Rainfall array (mm):")
print(R)

Shape: (5, 12)
Dtype: int64
Rainfall array (mm):
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


4. Annual totals + Monthly averages

In [4]:
annual_totals = R.sum(axis=1)
print("Annual total rainfall (mm):")
for county, total in zip(counties, annual_totals):
    print(f" {county}: {total}")

monthly_averages = R.mean(axis=0)
print("\nAverage rainfall per month across all counties (mm):")
for month, avg in zip(months, monthly_averages):
    print(f" {month}: {avg:.1f}")

Annual total rainfall (mm):
 Nairobi: 1045
 Nakuru: 916
 Kisumu: 1371
 Mombasa: 1075
 Machakos: 811

Average rainfall per month across all counties (mm):
 January: 47.0
 February: 47.6
 March: 91.0
 April: 164.4
 May: 171.2
 June: 67.6
 July: 56.6
 August: 58.2
 September: 64.2
 October: 67.8
 November: 127.4
 December: 80.6


5. Wettest month

In [5]:
wettest_idx = np.argmax(R, axis=1)
print("Wettest month per county:")
for county, idx in zip(counties, wettest_idx):
    print(f" {county}: {months[idx]}")

Wettest month per county:
 Nairobi: April
 Nakuru: April
 Kisumu: April
 Mombasa: May
 Machakos: November


6. Dry months

In [6]:
dry_mask = R < 50
dry_months = dry_mask.sum(axis=1)
print("Number of dry months (< 50 mm) per county:")
for county, count in zip(counties, dry_months):
    print(f" {county}: {count}")

Number of dry months (< 50 mm) per county:
 Nairobi: 4
 Nakuru: 3
 Kisumu: 0
 Mombasa: 2
 Machakos: 6


7. Planting window

In [7]:
consecutive = R[:, :-1] + R[:, 1:]
best_pair_idx = np.argmax(consecutive, axis=1)

print("Recommended planting windows (highest consecutive rainfall):")
planting_windows = []
for county, idx in zip(counties, best_pair_idx):
    window = f"{months[idx]}-{months[idx + 1]}"
    planting_windows.append(window)
    print(f" {county}: {window}")

Recommended planting windows (highest consecutive rainfall):
 Nairobi: April-May
 Nakuru: April-May
 Kisumu: April-May
 Mombasa: April-May
 Machakos: November-December


8. Write CSV

In [8]:
with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    for i, county in enumerate(counties):
        writer.writerow([
            county,
            int(annual_totals[i]),
            months[wettest_idx[i]],
            int(dry_months[i]),
            planting_windows[i]
        ])

print("rainfall_summary.csv written successfully.")

rainfall_summary.csv written successfully.


In [9]:
Planting Advisory

SyntaxError: invalid syntax (2565982757.py, line 1)

In [ ]:
Farmers in Nairobi, Nakuru, Kisumu and Mombasa should prioritise planting during the April–May window when consecutive rainfall peaks and soil moisture is most reliable. Machakos farmers should instead target the November–December period, which shows the strongest late-year rainfall combination. Across the five counties the long rains (March–May) remain the most consistent overall pattern, so early land preparation in March is advisable for most sites. Kisumu records the highest annual totals and no dry months, making it the most resilient for rain-fed crops, while Machakos has six dry months and the lowest annual total and therefore needs careful moisture conservation.

Data Quality Notes


In [ ]:
Two Machakos monthly values (April and September) were missing or invalid and were imputed with the county mean of 68 mm; this may slightly under- or over-state the true rainfall in those months and should be treated as approximate until station records can be verified.